# Step 01. The soft-thresholding power

weighted gene co-expression network analysis (WGCNA) raises every protein–protein correlation to a power, so weak correlations collapse toward
zero and strong ones survive. Everything downstream depends on this one number.

In [1]:
source("../src/paths.R")
suppressMessages(library(WGCNA))
options(stringsAsFactors = FALSE); enableWGCNAThreads(4)
COHORT <- "A"
X <- read.csv(coh("R_cohort-%s_log2_combat.csv", COHORT), row.names = 1, check.names = FALSE)
sprintf("%d patients x %d proteins", nrow(X), ncol(X))

Allowing parallel execution with up to 4 working processes.


[1] "87 patients x 7288 proteins"

In [2]:
sft <- pickSoftThreshold(X, powerVector = c(1:10, seq(12, 20, 2)),
                         networkType = "signed", verbose = 0)
sft$fitIndices[, c("Power", "SFT.R.sq", "slope", "mean.k.")]

   Power SFT.R.sq slope truncated.R.sq mean.k. median.k. max.k.
1      1    0.363 10.30          0.603  3710.0   3680.00   4070
2      2    0.310 -2.31          0.146  2040.0   1900.00   2680
3      3    0.635 -2.24          0.693  1200.0   1050.00   1970
4      4    0.771 -1.88          0.846   756.0    613.00   1550
5      5    0.827 -1.58          0.884   502.0    372.00   1280
6      6    0.865 -1.42          0.899   350.0    233.00   1080
7      7    0.889 -1.32          0.896   254.0    149.00    941
8      8    0.897 -1.26          0.884   192.0     97.50    829
9      9    0.905 -1.22          0.884   149.0     65.30    739
10    10    0.906 -1.19          0.880   119.0     44.60    664
11    12    0.912 -1.16          0.889    79.7     21.80    546
12    14    0.912 -1.14          0.899    56.6     11.30    457
13    16    0.882 -1.15          0.887    41.9      6.15    389
14    18    0.887 -1.15          0.904    32.0      3.46    335
15    20    0.862 -1.17          0.892  

Power,SFT.R.sq,slope,mean.k.
<dbl>,<dbl>,<dbl>,<dbl>
1,0.3633813,10.289489,3712.78116
2,0.3102615,-2.312635,2042.77504
3,0.6354193,-2.243911,1204.30759
4,0.7706295,-1.877109,755.75471
5,0.8270751,-1.584851,501.56980
6,0.8650417,-1.419512,349.67703
7,0.8894145,-1.316260,254.34669
8,0.8973456,-1.260631,191.76069
9,0.9053446,-1.215274,148.96228


## Signed, not unsigned

A signed network only joins proteins moving in the *same* direction. Unsigned would place a
protein and its mirror image in one module, which is not a co-expression module in any useful
sense.

This choice is why the automatic power estimate cannot be taken at face value. Signed adjacency
maps a correlation onto [0,1] by `(1+r)/2`, so an uncorrelated pair starts at 0.5, not 0, and
needs a much larger exponent before it is pushed away. `pickSoftThreshold` reports the first power
crossing its scale-free criterion, which for a signed network is reliably too permissive, at a low
power almost every pair stays connected and the modules come back as two or three lumps spanning
the whole panel.

The WGCNA authors publish a floor by sample count for signed networks, 18 below n=20, 16 to
n=30, 14 to n=40, 12 above, and it takes precedence over the estimate.

In [3]:
floor_power <- if (nrow(X) < 20) 18 else if (nrow(X) < 30) 16 else if (nrow(X) < 40) 14 else 12
power <- max(floor_power, ifelse(is.na(sft$powerEstimate), 0, sft$powerEstimate))
sprintf("estimate %s | floor for n=%d is %d | using %d (scale-free R2 = %.2f)",
        sft$powerEstimate, nrow(X), floor_power, power,
        sft$fitIndices$SFT.R.sq[sft$fitIndices$Power == power])

[1] "estimate 6 | floor for n=87 is 12 | using 12 (scale-free R2 = 0.91)"

On cohort A the estimate is 10 and the floor is 12; the fit at 12 is still R² = 0.89, so
nothing is given up by taking the floor.

This is a defensible choice, not a free one, and it should be stated in any write-up: the power
was set by the authors' published floor rather than by the automatic estimate, because the automatic
estimate on a signed network produced two modules spanning most of the panel.

## Where this was run

Rendered notebooks are committed, so each records the machine, the R and the
package versions that produced its output.


In [4]:
run_provenance()

run on   : Annes-MacBook-Pro-193.local ( Darwin 27.0.0 )
date     : 2026-09-25 12:02 EDT 
R        : R version 4.4.3 (2025-02-28) | x86_64-apple-darwin13.4.0 
R comes from: /Users/adeslatt/miniforge3/envs/endotypes-proteomics 
packages :
   WGCNA            1.74
   ComplexHeatmap   2.22.0
   sva              3.54.0
   VarSelLCM        2.1.3.2
   cluster          2.1.8.1
   fpc              2.2.15
   circlize         0.4.18
